# Tugas Akhir Praktikum Logical Agents: Wumpus World 5×5

## Identitas
- **Nama**   : Wayan Raditya Putra  
- **NRP**    : 5054241029  
- **Departemen** : Teknik Informatika  
- **Prodi** : RKA
- **Mata Kuliah** : Kecerdasan Komputasional
- **Dosen Pengampu** :  
  - Prof. Dr. Eng. Nanik Suciati, S.Kom., M.Kom.  
  - Imam Mustafa Kamal, S.ST, Ph.D.  

---

## Deskripsi Tugas
Agen ditempatkan pada lingkungan **Wumpus World 5×5**.  
- Posisi awal agen: `[1,1]`  
- Posisi emas: `[2,4]`  
- Posisi pit: `(1,3), (1,5), (4,1), (4,5)`  
- Posisi Wumpus: `(3,1)`  

Aturan Wumpus World:  
- Pit menimbulkan **breeze** di sel tetangga (atas, bawah, kiri, kanan).  
- Wumpus menimbulkan **stench** di sel tetangga (atas, bawah, kiri, kanan).  

---

## Tujuan
1. Mendefinisikan proposisi dan aturan logika (R1–Rn) berdasarkan aturan Wumpus World.  
2. Menyusun rangkaian proposisi secara sistematis dari posisi awal hingga emas.  
3. Melakukan inferensi menggunakan **Truth Table Entailment (TT-entails)** untuk memverifikasi kebenaran inferensi.  
4. Melakukan inferensi menggunakan **Forward Chaining** (NRP ganjil).  
5. Membuktikan apakah agen dapat mencapai emas dengan aman, serta menuliskan jalur langkah demi langkah.  
6. Mengimplementasikan program Python yang merepresentasikan inferensi logika agen.  
---

## Pendahuluan

Pada bidang **Kecerdasan Buatan (Artificial Intelligence)**, salah satu pendekatan yang digunakan untuk 
merepresentasikan pengetahuan adalah melalui **Logical Agents**. Logical agent merupakan agen yang 
mengambil keputusan berdasarkan **aturan logika** dan inferensi, bukan sekadar trial-and-error.  
Agen jenis ini cocok untuk lingkungan yang penuh ketidakpastian tetapi memiliki aturan formal yang 
jelas, seperti **Wumpus World**.

**Wumpus World** adalah dunia berbentuk grid yang dipopulerkan dalam literatur AI (Russell & Norvig). 
Lingkungan ini digunakan untuk menguji kemampuan agen dalam bernavigasi secara aman untuk mencapai 
tujuan (misalnya menemukan emas), sambil menghindari bahaya berupa **pit** (lubang) dan **wumpus** 
(monster). Agen hanya bisa merasakan indikator lingkungan:  
- **Breeze** → muncul di sel yang berdekatan dengan pit.  
- **Stench** → muncul di sel yang berdekatan dengan wumpus.  

Untuk dapat bernavigasi dengan aman, agen harus mampu:  
1. Mendefinisikan proposisi yang merepresentasikan kondisi dunia.  
2. Menggunakan aturan logika (R1–Rn) untuk melakukan inferensi.  
3. Menentukan langkah berikutnya berdasarkan inferensi yang sahih.

Dalam tugas ini digunakan dua metode inferensi utama:  
- **Truth Table Entailment (TT-entails)**: metode dasar dengan membangun tabel kebenaran untuk 
  memverifikasi apakah suatu proposisi logis benar secara konsisten.  
- **Forward Chaining**: metode berbasis aturan produksi yang mulai dari fakta yang diketahui 
  kemudian menurunkan fakta-fakta baru sampai mencapai kesimpulan.  

Dengan pendekatan ini, agen diharapkan dapat membuktikan apakah emas di koordinat `(2,4)` dapat 
dicapai dari posisi awal `(1,1)` dengan aman sesuai aturan Wumpus World.


## Definisi Proposisi & Aturan Umum (R1–Rn)

### Proposisi
1. **P(x,y)** : Ada *pit* pada sel `(x,y)`  
2. **W(x,y)** : Ada *wumpus* pada sel `(x,y)`  
3. **G(x,y)** : Ada emas (*gold*) pada sel `(x,y)`  
4. **B(x,y)** : Ada *breeze* pada sel `(x,y)`  
5. **S(x,y)** : Ada *stench* pada sel `(x,y)`  
6. **OK(x,y)** : Sel `(x,y)` aman untuk dimasuki agen  
7. **¬OK(x,y)** : Sel `(x,y)` berbahaya (ada kemungkinan pit atau wumpus) 

---

### Aturan Umum (R1–Rn)
**R1 (Breeze Rule)**  
B(x,y) ↔ (P(x-1,y) ∨ P(x+1,y) ∨ P(x,y-1) ∨ P(x,y+1))

**R2 (Stench Rule)**  
S(x,y) ↔ (W(x-1,y) ∨ W(x+1,y) ∨ W(x,y-1) ∨ W(x,y+1))
  
**R3 (Safety Rule)**  
  ¬B(x,y) ∧ ¬S(x,y) → OK(x,y)
  
**R4 (Pit Inference)**  
Jika ada breeze di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti pit.  

**Notasi:**
B(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → P(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  

**R5 (Wumpus Inference)**  
Jika ada stench di `(x,y)` dan semua tetangga selain `(i,j)` sudah terbukti aman,  
maka tetangga `(i,j)` pasti wumpus.  

**Notasi:**
S(x,y) ∧ (OK(t₁) ∧ OK(t₂) ∧ ... ∧ OK(tₙ)) → W(i,j)  

dengan:  
- `(t₁, t₂, ..., tₙ)` = semua tetangga dari `(x,y)` **kecuali** `(i,j)`  
- `(i,j)` = satu-satunya tetangga yang belum aman  


**R6 (Gold Detection)**  
Jika ada emas di `(x,y)`, maka `(x,y)` adalah tujuan agen.  

**Notasi:**
G(x,y) → Goal(x,y)

## Project Setup

Pada tahap ini dilakukan proses **import library** dan modul pendukung yang dibutuhkan untuk mengerjakan tugas Logical Agents pada Wumpus World.  
Library yang digunakan terdiri dari modul internal (`utils.py`, `logic.py`, `agents.py`) maupun library eksternal Python.

### Penjelasan Library

* **utils** → berisi fungsi utilitas tambahan yang mendukung proses inferensi/logika.
* **logic** → berisi implementasi representasi logika (proposisi, aturan, inferensi).
* **agents** → modul berisi definisi agen dan cara agen berinteraksi dengan Wumpus World.
* **math** → library standar Python untuk operasi matematika.
* **inspect.getsource** → digunakan untuk menampilkan kode sumber dari fungsi tertentu (berguna saat analisis).
* **IPython.display.HTML** → menampilkan output HTML di Jupyter Notebook.
* **tabulate** → menghasilkan tabel rapi untuk menyajikan hasil inferensi dan analisis.



In [6]:
from utils import *
from logic import *
import agents
import math
from inspect import getsource
from IPython.display import HTML
from tabulate import tabulate

## Representasi Peta dan Informasi Wumpus World

Kelas `WumpusAgent` digunakan sebagai **alat bantu** untuk membangun representasi dunia Wumpus 
dalam bentuk grid (map) serta memasukkan informasi lingkungan ke dalam *knowledge base* (KB) agen.

- **World (Peta 5×5)** direpresentasikan sebagai list of list, di mana setiap cell dapat berisi:
  - `"P"` → Pit
  - `"W"` → Wumpus
  - `"G"` → Gold
  - `"B"` → Breeze
  - `"S"` → Stench
  - `[]`   → Kosong (tidak ada percept)

- **Knowledge Base (KB)** dibangun secara bertahap:
  1. Agen memulai dari posisi awal `(1,1)` → otomatis dianggap aman (`Safe(1,1)`).
  2. Agen membaca **percepts** di cell tersebut (misalnya Breeze, Stench).
  3. Informasi percepts dimasukkan ke dalam KB sebagai proposisi logika, contohnya:
     - `Breeze(1,2)`
     - `Stench(2,1)`
     - `Gold(2,4)`

Dengan cara ini, agen dapat menghubungkan **peta (map)** dan **pengetahuan logika** yang 
dibutuhkan untuk melakukan inferensi, sehingga jalur menuju emas dapat ditentukan secara aman.


In [25]:

# Sumber = Aima Data, Class wumpus , agent, 
class WumpusAgent:
    def __init__(self, world, initial_pos=(1,1)):
        self.world = world
        self.kb = PropKB()
        self.visited = set()
        self.position = initial_pos

        # Cell Agent pertama kali
        self.kb.tell(expr(f"Safe({initial_pos[0]},{initial_pos[1]})"))
        self.visited.add(initial_pos)

        # Update KB berdasarkan percepts
        self.update_kb(*initial_pos)

    def get_percepts(self, x, y):
        """
        Ambil percept dari koordinat (x,y) sesuai world
        """
        world_x = len(self.world) - y
        world_y = x - 1
        if 0 <= world_x < len(self.world) and 0 <= world_y < len(self.world[0]):
            return set(self.world[world_x][world_y])
        return set()

    # … di dalam class WumpusAgent yang sudah ada …

        # … di dalam class WumpusAgent yang sudah ada …

    def update_kb(self, x, y):
        """
        Update KB dari percepts posisi (x,y):
        - Simpan Breeze/Stench/Gold jika ada
        - Simpan NoBreeze/NoStench jika tidak ada (penting untuk inferensi)
        - Tandai sel saat ini aman (agen berada di situ & tidak mati)
        """
        percepts = self.get_percepts(x, y)
        hasB = ("B" in percepts) or ("Breeze" in percepts)
        hasS = ("S" in percepts) or ("Stench" in percepts)
        hasG = ("G" in percepts) or ("Gold" in percepts)

        if hasB: self.kb.tell(expr(f"Breeze({x},{y})"))
        else:    self.kb.tell(expr(f"NoBreeze({x},{y})"))

        if hasS: self.kb.tell(expr(f"Stench({x},{y})"))
        else:    self.kb.tell(expr(f"NoStench({x},{y})"))

        if hasG: self.kb.tell(expr(f"Gold({x},{y})"))

        # Agen hidup di sel ini ⇒ sel ini aman
        self.kb.tell(expr(f"Safe({x},{y})"))
        self.visited.add((x,y))


In [26]:
world = [
    # y = 5
    [["P"], ["B"], ["B"], ["P"], ["B"]],   # row 5
    # y = 4
    [["B"], ["G"], [], ["B"], []],      # row 4
    # y = 3
    [["P"], [], [], [], []],            # row 3
    # y = 2
    [["B"], [], ["S"], ["B"], []],         # row 2
    # y = 1
    [[], ["S"], ["W"], ["P"], ["B"]]     # row 1
]

agent = WumpusAgent(world, initial_pos=(1,1))

# # world: list of list, baris paling atas = y=5, baris paling bawah = y=1
# rows = len(world)      # jumlah baris (y)
# cols = len(world[0])   # jumlah kolom (x)

# # Loop semua koordinat dan masukan semua percpet ke kb agent
# for row in range(rows):
#     for col in range(cols):
#         x = col + 1              # kolom ke-x (1-based)
#         y = rows - row           # konversi: row=0 → y=5, row=4 → y=1
#         agent.update_kb(x, y)




In [27]:
from functools import reduce
from operator import and_ as AND
from logic import expr, tt_entails  # pl_fc_entails opsional; kita fallback kalau tak ada
import time

WORLD_N = 5

def neighbors(x, y, n=WORLD_N):
    cand = [(x+1,y),(x-1,y),(x,y+1),(x,y-1)]
    return [(i,j) for (i,j) in cand if 1 <= i <= n and 1 <= j <= n]

def conjoin(exprs):
    # Gabungkan list of Expr menjadi satu Expr dengan AND (untuk TT-entails)
    if not exprs:
        return expr("True")
    return reduce(lambda a,b: a & b, exprs)

def add_rules_to_kb(kb):
    """
    Tambahkan aturan umum (Horn clauses) ke KB:
    - NoBreeze & NoStench pada (x,y) ⇒ Safe tetangga
    - NoBreeze pada (x,y) ⇒ NoPit tetangga
    - NoStench pada (x,y) ⇒ NoW tetangga
    - NoPit & NoW pada (i,j) ⇒ Safe(i,j)
    - Uniqueness:
      Breeze(x,y) & NoPit(tetangga lain…) ⇒ P(tetangga sisa)
      Stench(x,y) & NoW(tetangga lain…)  ⇒ W(tetangga sisa)
    """
    for x in range(1, WORLD_N+1):
        for y in range(1, WORLD_N+1):
            ngh = neighbors(x,y)
            for (i,j) in ngh:
                kb.tell(expr(f"NoBreeze({x},{y}) & NoStench({x},{y}) ==> Safe({i},{j})"))
                kb.tell(expr(f"NoBreeze({x},{y}) ==> NoPit({i},{j})"))
                kb.tell(expr(f"NoStench({x},{y}) ==> NoW({i},{j})"))
                kb.tell(expr(f"NoPit({i},{j}) & NoW({i},{j}) ==> Safe({i},{j})"))
            # Uniqueness rules (satu kandidat tersisa)
            if len(ngh) >= 1:
                for (i,j) in ngh:
                    others_np = " & ".join([f"NoPit({a},{b})" for (a,b) in ngh if (a,b)!=(i,j)])
                    premise_p = f"Breeze({x},{y})" + (f" & {others_np}" if others_np else "")
                    kb.tell(expr(f"{premise_p} ==> P({i},{j})"))
                    others_nw = " & ".join([f"NoW({a},{b})" for (a,b) in ngh if (a,b)!=(i,j)])
                    premise_w = f"Stench({x},{y})" + (f" & {others_nw}" if others_nw else "")
                    kb.tell(expr(f"{premise_w} ==> W({i},{j})"))


In [28]:
def entails_safe(agent, nx, ny, mode="fc"):
    q = expr(f"Safe({nx},{ny})")
    if mode == "tt":
        kb_expr = conjoin(agent.kb.clauses)
        return tt_entails(kb_expr, q)
    elif mode == "fc":
        # Coba pl_fc_entails kalau tersedia; kalau tidak, fallback:
        try:
            from logic import pl_fc_entails
            return pl_fc_entails(agent.kb, q)
        except Exception:
            # fallback: gunakan hasil forward_chain_procedural
            return q in agent.kb.clauses
    else:
        raise ValueError("mode harus 'tt' atau 'fc'")


In [29]:
def forward_chain_procedural(agent):
    """
    Derivasi fakta baru dari KB berbasis rules dasar (tanpa bergantung ke pl_fc_entails).
    Ini menjalankan beberapa pass sampai tidak ada fakta baru yang bisa ditambahkan.
    """
    changed = True
    while changed:
        changed = False
        # Pakai snapshot supaya looping aman saat KB bertambah
        visited_now = list(agent.visited)
        for (x,y) in visited_now:
            # If NoBreeze/NoStench di (x,y), turunkan NoPit/NoW ke semua tetangga
            if expr(f"NoBreeze({x},{y})") in agent.kb.clauses:
                for (i,j) in neighbors(x,y):
                    e = expr(f"NoPit({i},{j})")
                    if e not in agent.kb.clauses:
                        agent.kb.tell(e); changed = True
            if expr(f"NoStench({x},{y})") in agent.kb.clauses:
                for (i,j) in neighbors(x,y):
                    e = expr(f"NoW({i},{j})")
                    if e not in agent.kb.clauses:
                        agent.kb.tell(e); changed = True

            # Uniqueness untuk Pit
            if expr(f"Breeze({x},{y})") in agent.kb.clauses:
                cand = neighbors(x,y)
                unknown = [(i,j) for (i,j) in cand
                           if expr(f"NoPit({i},{j})") not in agent.kb.clauses
                           and expr(f"Safe({i},{j})")   not in agent.kb.clauses]
                if len(unknown) == 1:
                    (i,j) = unknown[0]
                    e = expr(f"P({i},{j})")
                    if e not in agent.kb.clauses:
                        agent.kb.tell(e); changed = True

            # Uniqueness untuk Wumpus
            if expr(f"Stench({x},{y})") in agent.kb.clauses:
                cand = neighbors(x,y)
                unknown = [(i,j) for (i,j) in cand
                           if expr(f"NoW({i},{j})")  not in agent.kb.clauses
                           and expr(f"Safe({i},{j})") not in agent.kb.clauses]
                if len(unknown) == 1:
                    (i,j) = unknown[0]
                    e = expr(f"W({i},{j})")
                    if e not in agent.kb.clauses:
                        agent.kb.tell(e); changed = True

            # Jika NoPit & NoW pada sel tetangga ⇒ Safe
            for (i,j) in neighbors(x,y):
                if (expr(f"NoPit({i},{j})") in agent.kb.clauses and
                    expr(f"NoW({i},{j})")   in agent.kb.clauses):
                    e = expr(f"Safe({i},{j})")
                    if e not in agent.kb.clauses:
                        agent.kb.tell(e); changed = True


In [30]:
def explore(agent, mode="fc"):
    """
    Eksplorasi tanpa 'bocoran goal':
    - Agen bergerak hanya ke sel yang TERBUKTI aman via mode entailment.
    - Berhenti saat sensor Gold terdeteksi pada sel yang ditempati.
    - Mengukur waktu eksekusi.
    """
    t0 = time.time()
    path = [agent.position]

    # pastikan aturan ter-install
    # (aman kalau dipanggil berulang; aturan yang sama tidak merusak KB)
    add_rules_to_kb(agent.kb)

    while True:
        x, y = agent.position

        # update KB dari posisi saat ini
        agent.update_kb(x, y)

        # jalankan FC prosedural agar fakta turunan (NoPit/NoW/Safe) bertambah
        forward_chain_procedural(agent)

        # jika menemukan gold, berhenti
        if "G" in agent.get_percepts(x, y) or expr(f"Gold({x},{y})") in agent.kb.clauses:
            print(f"[{mode.upper()}] GOLD ditemukan di {x,y}")
            break

        # pilih tetangga yang belum dikunjungi
        cand = [c for c in neighbors(x,y) if c not in agent.visited]

        moved = False
        for (nx, ny) in cand:
            if entails_safe(agent, nx, ny, mode=mode):
                # Move
                agent.position = (nx, ny)
                agent.visited.add((nx, ny))
                agent.kb.tell(expr(f"Safe({nx},{ny})"))
                path.append((nx, ny))
                print(f"[{mode.upper()}] → move ke {(nx,ny)} (terbukti aman)")
                moved = True
                break

        if not moved:
            print(f"[{mode.upper()}] Tidak ada tetangga yang TERBUKTI aman. Stop.")
            break

    dt = time.time() - t0
    print(f"Waktu eksekusi ({mode.upper()}): {dt:.6f} detik")
    return path


In [31]:
# (1) Buat ulang agent dari world yang sudah lo definisikan
agent = WumpusAgent(world, initial_pos=(1,1))

# (2) Eksplorasi pakai Forward Chaining
path_fc = explore(agent, mode="fc")
print("Path (FC):", path_fc)

# (3) Reset agent untuk TT
agent = WumpusAgent(world, initial_pos=(1,1))
path_tt = explore(agent, mode="tt")
print("Path (TT):", path_tt)


[FC] → move ke (2, 1) (terbukti aman)
[FC] Tidak ada tetangga yang TERBUKTI aman. Stop.
Waktu eksekusi (FC): 0.025183 detik
Path (FC): [(1, 1), (2, 1)]


KeyboardInterrupt: 

In [ ]:
# %%timeit # buat ukur waktu program
# # Gabungkan semua klausa di KB
# kb_expr = conjoin(agent.kb.clauses)
# print("Knowledge Base:", kb_expr)

# # Daftar proposisi penting yang mau dicek (Pareto 20%)
# queries = [
#     expr("Safe(1,1)"),
#     expr("Safe(1,2)"),
#     expr("Safe(2,1)"),
#     expr("P(1,3)"),
#     expr("W(3,1)"),
#     expr("G(2,2)"),   # cek apakah ada gold
# ]

# # Cek satu per satu
# for q in queries:
#     result = tt_entails(kb_expr, q)
#     print(f"Apakah {q} ter-entail oleh KB? , Jawaban : {result}")


In [ ]:
print()

In [ ]:
# # inferensi dengan forward chainning
# # Gabungkan semua klausa di KB
# kb_expr = conjoin(agent.kb.clauses)
# print("Knowledge Base:", kb_expr)

# # Daftar proposisi penting yang mau dicek (Pareto 20%)
# queries = [
#     expr("Safe(1,1)"),
#     expr("Safe(1,2)"),
#     expr("Safe(2,1)"),
#     expr("P(1,3)"),
#     expr("W(3,1)"),
#     expr("G(2,2)"),   # cek apakah ada gold
# ]

# # Cek satu per satu
# for q in queries:
#     result = pl_fc_entails(kb_expr, q)
#     print(f"Apakah {q} ter-entail oleh KB? , Jawaban : {result}")
